In [44]:
import pandas as pd
import os
import ast
import re
import warnings
warnings.filterwarnings('ignore')

In [45]:
list_name = os.listdir('./')

In [71]:
def process_string_to_dict(string):
    string = string.split('; Count')[0]
    # Substituir os `;` por vírgulas e remover espaços extras
    formatted_string = re.sub(r";", ",", string)
    formatted_string = re.sub(r"=", ":", formatted_string)
    # Adicionar chaves externas para transformá-la em um dicionário
    formatted_string = "{" + formatted_string + "}"
    # Substituir chaves como "Code.or.Value" por "Code" e "Value.Description" por "Description"
    formatted_string = re.sub(r"Code\.or\.Value", '"Code"', formatted_string)
    formatted_string = re.sub(r"Value\.Description", '"Description"', formatted_string)

    return eval(formatted_string)



list_name_types = ['demo', 'diet', 'exam', 'lab', 'q']
list_names_file = ['DEMO_tabelas_lab_completas_resultados.csv', 'DIET_tabelas_lab_completas_resultados.csv', 'EXAM_tabelas_lab_completas_resultados.csv',
                   'LAB_tabelas_lab_completas_resultados.csv', 'Q_tabelas_lab_completas_resultados.csv']

for idx, type in enumerate(list_name_types):
    file = list_names_file[idx]
    tab_main = pd.read_csv(f'./{file}')
    
    dict_list = tab_main.apply(lambda row: {'name': row['Data.File.Name'], 'begin': str(row['Begin.Year']), 'end': str(row['EndYear'])}, axis=1).tolist()
    
    table_names = tab_main['Data.File.Name'].tolist()
    
    
    for current_name in table_names:
        print(f"***************************{current_name}************************")
        try:
        
            df_now = pd.read_csv(f'./{type}/{type.upper()}-{current_name.upper()}.csv')
            resultado = [d for d in dict_list if d['name'] == current_name]
            
            df_sem_duplicatas = df_now.drop_duplicates()
            codebook_select_col = df_sem_duplicatas[['Tabela','Variable_Name', 'SAS_Label', 'Tabela_Valores','English_Text', 'Target', 'English_Instructions']]
            list_Variable_Name = codebook_select_col['Variable_Name'].unique()
            
            df_codebook_final = pd.DataFrame()
            for variable in list_Variable_Name:
                df_now = codebook_select_col[codebook_select_col['Variable_Name']==variable]
                list_english = list(df_now['English_Text'].unique())
                list_Target = list(df_now['Target'].unique())
                list_English_Instructions = list(df_now['English_Instructions'].unique())
                df_now['English_Text'] = str(list_english[0])
                df_now['Target'] = str(list_Target[0])
                df_now['English_Instructions'] = str(list_English_Instructions[0])

                df_codebook_final = pd.concat([df_codebook_final, df_now])
                
            rows = []
            for _, row in df_codebook_final.iterrows():
                # Ignorar linhas onde Tabela_Valores é NaN
                if pd.isna(row['Tabela_Valores']):
                    rows.append({
                                "Column": row["Variable_Name"],
                                "Code": None,
                                "Label": None,
                                "English_Text": row["English_Text"],
                                "Target": row["Target"],
                                "English_Instructions": row["English_Instructions"]
                            })
                    continue
                
                # Extrair os dados de Code e Value.Description
                valores = row['Tabela_Valores']
                valores = re.sub(r'\s+', ' ', valores)
                valores = re.sub(r':\s+\[', ': [', valores)
                
                try:
                    # Usando ast.literal_eval para transformar a string em dicionário
                    parsed_values = process_string_to_dict(valores)
                    codes = parsed_values["Code"]
                    codes = [str(item).title() for item in codes]
                    descriptions = parsed_values["Description"]
                    descriptions = [str(item).title() for item in descriptions]
                    
                    # Criar uma nova linha para cada combinação de Code e Description
                    for code, description in zip(codes, descriptions):
                        rows.append({
                            "Column": row["Variable_Name"],
                            "Code": code,
                            "Label": description,
                            "English_Text": row["English_Text"],
                            "Target": row["Target"],
                            "English_Instructions": row["English_Instructions"]
                        })
                        
                except Exception as e:
                    print(f"Erro ao processar a linha: {row['Tabela_Valores']}")
                    print(e)

            # Criar um novo DataFrame com os dados processados
            df_expanded = pd.DataFrame(rows)
            df_expanded['Class'] = None
            df_expanded['Other For'] = None
                
            df_expanded.drop_duplicates(inplace=True)

            caminho = f"{type}/SDDs"

            # Verifica se a pasta existe, se não, cria
            if not os.path.exists(caminho):
                os.makedirs(caminho)
                print(f"Pasta criada: {caminho}")
            else:
                print(f"A pasta já existe: {caminho}")


            print(resultado[0]['begin'])
            begin = resultado[0]['begin']
            end = resultado[0]['end']
            df_expanded.to_csv(f'./{type}/SDDs/CODEBOOK-NHANES-{begin}-{end}-{current_name}.csv', sep=',', encoding='utf-8')
            
            
            
            dictionary_mapping = codebook_select_col[['Variable_Name']]
            dictionary_mapping = dictionary_mapping.drop_duplicates(subset='Variable_Name', keep='first')

            dictionary_mapping['Attribute'] = None
            dictionary_mapping['attributeOf'] = None
            dictionary_mapping['Unit'] = None
            dictionary_mapping['Time'] = None
            dictionary_mapping['Entity'] = None
            dictionary_mapping['Role'] = None
            dictionary_mapping['Relation'] = None
            dictionary_mapping['inRelationTo'] = None
            dictionary_mapping['wasDerivedFrom'] = None
            dictionary_mapping['Maturity'] = None
            dictionary_mapping['Template'] = None
            dictionary_mapping['wasGeneratedBy'] = None
            dictionary_mapping.reset_index(inplace=True, drop=True)
            dictionary_mapping.rename(columns={"Variable_Name": "Column"}, inplace=True)
            
            dictionary_mapping.to_csv(f'./{type}/SDDs/dictionary_mapping-NHANES-{begin}-{end}-{current_name}.csv', sep=',', encoding='utf-8')
            
            
            with pd.ExcelWriter(f'./{type}/SDDs/SDD-NHANES-{begin}-{end}-{current_name}.xlsx', engine='xlsxwriter') as writer:
                df_expanded.to_excel(writer, sheet_name='codebook', index=False)
                dictionary_mapping.to_excel(writer, sheet_name='dictionary_mapping', index=False)
                
        except Exception as e:
            print(f"Erro ao processar a tabela {current_name}")


***************************DEMO************************
A pasta já existe: demo/SDDs
1999
***************************DEMO_B************************
A pasta já existe: demo/SDDs
2001
***************************DEMO_C************************
A pasta já existe: demo/SDDs
2003
***************************DEMO_D************************
A pasta já existe: demo/SDDs
2005
***************************DEMO_E************************
A pasta já existe: demo/SDDs
2007
***************************DEMO_F************************
A pasta já existe: demo/SDDs
2009
***************************DEMO_G************************
A pasta já existe: demo/SDDs
2011
***************************DEMO_H************************
A pasta já existe: demo/SDDs
2013
***************************DEMO_I************************
A pasta já existe: demo/SDDs
2015
***************************DEMO_J************************
A pasta já existe: demo/SDDs
2017
***************************P_DEMO************************
A pasta já existe: demo/

In [39]:
resultado[0]['begin']

'2021'

In [51]:
tab_main_MSX = pd.read_csv('./exam/MSX.csv')

FileNotFoundError: [Errno 2] No such file or directory: './exam/MSX.csv'

In [ ]:
df_now = pd.read_csv(f'./{type}/{type.upper()}-{current_name.upper()}.csv')

In [72]:
tab_demo = pd.read_csv('./q/Q-P_DBQ.csv')
tab_demo

,Tabela,Variavel,Variable_Name,SAS_Label,English_Text,Target,English_Instructions,Tabela_Valores
0,P_DBQ,CBQ596,CBQ596,Heard of My Plate,Next Im going to ask a few questions about the...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
1,P_DBQ,CBQ606,CBQ606,Looked up My Plate on internet,{Have you/Has SP} looked up the My Plate plan ...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
2,P_DBQ,CBQ611,CBQ611,Tried My Plate plan,{Have you/Has SP} tried to follow the recommen...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
3,P_DBQ,DBD030,DBD030,Age stopped breastfeeding(days),How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
4,P_DBQ,DBD041,DBD041,Age first fed formula(days),How old was {SP} when {he/she} was first fed f...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 365"", ""0"", ""666666"", ""777..."
5,P_DBQ,DBD050,DBD050,Age stopped receiving formula(days),How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
6,P_DBQ,DBD055,DBD055,Age started other food/beverage,This next question is about the first thing th...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 730"", ""0"", ""666666"", ""777..."
7,P_DBQ,DBD061,DBD061,Age first fed milk(days),How old was {SP} when {he/she} was first fed m...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
8,P_DBQ,DBD381,DBD381,# of times/week get school lunch,During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ..."
9,P_DBQ,DBD411,DBD411,# of times/week get school breakfast,During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ..."


In [12]:
tab_demo = pd.read_csv('./demo/DEMO-DEMO.csv')
tab_demo

,Tabela,Variavel,Variable_Name,SAS_Label,English_Text,Target,English_Instructions,Tabela_Valores
0,DEMO,SEQN,SEQN,Respondent sequence number,Respondent sequence number.,Both males and females 0 YEARS - 150 YEARS,NaN,NaN
1,DEMO,SDDSRVYR,SDDSRVYR,Data Release Number,Data Release Number.,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", "".""]; Value.Description=[""..."
2,DEMO,RIDSTATR,RIDSTATR,Interview/Examination Status,Interview and Examination Status of the Sample...,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", "".""]; Value.Descripti..."
3,DEMO,RIDEXMON,RIDEXMON,Six month time period,Six month time period when the examination was...,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", "".""]; Value.Descripti..."
4,DEMO,RIAGENDR,RIAGENDR,Gender,Gender of the sample person,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", "".""]; Value.Descripti..."
...,...,...,...,...,...,...,...,...
139,DEMO,WTIREP48,WTIREP48,Interview Weight Jack Knife Replicate 48,Interview Weight Jack Knife Replicate 48,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""0 to 237147.54679"", "".""]; Valu..."
140,DEMO,WTIREP49,WTIREP49,Interview Weight Jack Knife Replicate 49,Interview Weight Jack Knife Replicate 49,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""0 to 237076.10369"", "".""]; Valu..."
141,DEMO,WTIREP50,WTIREP50,Interview Weight Jack Knife Replicate 50,Interview Weight Jack Knife Replicate 50,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""0 to 238360.24085"", "".""]; Valu..."
142,DEMO,WTIREP51,WTIREP51,Interview Weight Jack Knife Replicate 51,Interview Weight Jack Knife Replicate 51,Both males and females 0 YEARS - 150 YEARS,NaN,"Code.or.Value=[""0 to 236648.97986"", "".""]; Valu..."


In [73]:
df_sem_duplicatas = tab_demo.drop_duplicates()
df_sem_duplicatas

,Tabela,Variavel,Variable_Name,SAS_Label,English_Text,Target,English_Instructions,Tabela_Valores
0,P_DBQ,CBQ596,CBQ596,Heard of My Plate,Next Im going to ask a few questions about the...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
1,P_DBQ,CBQ606,CBQ606,Looked up My Plate on internet,{Have you/Has SP} looked up the My Plate plan ...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
2,P_DBQ,CBQ611,CBQ611,Tried My Plate plan,{Have you/Has SP} tried to follow the recommen...,Both males and females 16 YEARS - 150 YEARS,NaN,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value..."
3,P_DBQ,DBD030,DBD030,Age stopped breastfeeding(days),How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
4,P_DBQ,DBD041,DBD041,Age first fed formula(days),How old was {SP} when {he/she} was first fed f...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 365"", ""0"", ""666666"", ""777..."
5,P_DBQ,DBD050,DBD050,Age stopped receiving formula(days),How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
6,P_DBQ,DBD055,DBD055,Age started other food/beverage,This next question is about the first thing th...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 730"", ""0"", ""666666"", ""777..."
7,P_DBQ,DBD061,DBD061,Age first fed milk(days),How old was {SP} when {he/she} was first fed m...,Both males and females 0 YEARS - 6 YEARS,NaN,"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77..."
8,P_DBQ,DBD381,DBD381,# of times/week get school lunch,During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ..."
9,P_DBQ,DBD411,DBD411,# of times/week get school breakfast,During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ..."


In [74]:
codebook_select_col = df_sem_duplicatas[['Tabela','Variable_Name', 'SAS_Label', 'Tabela_Valores','English_Text', 'Target', 'English_Instructions']]

In [75]:
codebook_select_col

,Tabela,Variable_Name,SAS_Label,Tabela_Valores,English_Text,Target,English_Instructions
0,P_DBQ,CBQ596,Heard of My Plate,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",Next Im going to ask a few questions about the...,Both males and females 16 YEARS - 150 YEARS,NaN
1,P_DBQ,CBQ606,Looked up My Plate on internet,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",{Have you/Has SP} looked up the My Plate plan ...,Both males and females 16 YEARS - 150 YEARS,NaN
2,P_DBQ,CBQ611,Tried My Plate plan,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",{Have you/Has SP} tried to follow the recommen...,Both males and females 16 YEARS - 150 YEARS,NaN
3,P_DBQ,DBD030,Age stopped breastfeeding(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN
4,P_DBQ,DBD041,Age first fed formula(days),"Code.or.Value=[""1 to 365"", ""0"", ""666666"", ""777...",How old was {SP} when {he/she} was first fed f...,Both males and females 0 YEARS - 6 YEARS,NaN
5,P_DBQ,DBD050,Age stopped receiving formula(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",How old was {SP} when {he/she} completely stop...,Both males and females 0 YEARS - 6 YEARS,NaN
6,P_DBQ,DBD055,Age started other food/beverage,"Code.or.Value=[""1 to 730"", ""0"", ""666666"", ""777...",This next question is about the first thing th...,Both males and females 0 YEARS - 6 YEARS,NaN
7,P_DBQ,DBD061,Age first fed milk(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",How old was {SP} when {he/she} was first fed m...,Both males and females 0 YEARS - 6 YEARS,NaN
8,P_DBQ,DBD381,# of times/week get school lunch,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ...",During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN
9,P_DBQ,DBD411,# of times/week get school breakfast,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ...",During the school year about how many times a ...,Both males and females 4 YEARS - 19 YEARS,NaN


In [76]:
list_Variable_Name = codebook_select_col['Variable_Name'].unique()
df_codebook_final = pd.DataFrame()

for variable in list_Variable_Name:
    df_now = codebook_select_col[codebook_select_col['Variable_Name']==variable]
    list_english = list(df_now['English_Text'].unique())
    list_Target = list(df_now['Target'].unique())
    list_English_Instructions = list(df_now['English_Instructions'].unique())
    df_now['English_Text'] = str(list_english)
    df_now['Target'] = str(list_Target)
    df_now['English_Instructions'] = str(list_English_Instructions)

    df_codebook_final = pd.concat([df_codebook_final, df_now])


In [77]:
df_codebook_final

,Tabela,Variable_Name,SAS_Label,Tabela_Valores,English_Text,Target,English_Instructions
0,P_DBQ,CBQ596,Heard of My Plate,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",['Next Im going to ask a few questions about t...,['Both males and females 16 YEARS - 150 YEARS'],[nan]
1,P_DBQ,CBQ606,Looked up My Plate on internet,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",['{Have you/Has SP} looked up the My Plate pla...,['Both males and females 16 YEARS - 150 YEARS'],[nan]
2,P_DBQ,CBQ611,Tried My Plate plan,"Code.or.Value=[""1"", ""2"", ""7"", ""9"", "".""]; Value...",['{Have you/Has SP} tried to follow the recomm...,['Both males and females 16 YEARS - 150 YEARS'],[nan]
3,P_DBQ,DBD030,Age stopped breastfeeding(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",['How old was {SP} when {he/she} completely st...,['Both males and females 0 YEARS - 6 YEARS'],[nan]
4,P_DBQ,DBD041,Age first fed formula(days),"Code.or.Value=[""1 to 365"", ""0"", ""666666"", ""777...",['How old was {SP} when {he/she} was first fed...,['Both males and females 0 YEARS - 6 YEARS'],[nan]
5,P_DBQ,DBD050,Age stopped receiving formula(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",['How old was {SP} when {he/she} completely st...,['Both males and females 0 YEARS - 6 YEARS'],[nan]
6,P_DBQ,DBD055,Age started other food/beverage,"Code.or.Value=[""1 to 730"", ""0"", ""666666"", ""777...",['This next question is about the first thing ...,['Both males and females 0 YEARS - 6 YEARS'],[nan]
7,P_DBQ,DBD061,Age first fed milk(days),"Code.or.Value=[""1 to 1095"", ""0"", ""666666"", ""77...",['How old was {SP} when {he/she} was first fed...,['Both males and females 0 YEARS - 6 YEARS'],[nan]
8,P_DBQ,DBD381,# of times/week get school lunch,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ...",['During the school year about how many times ...,['Both males and females 4 YEARS - 19 YEARS'],[nan]
9,P_DBQ,DBD411,# of times/week get school breakfast,"Code.or.Value=[""1 to 5"", ""0"", ""7777"", ""9999"", ...",['During the school year about how many times ...,['Both males and females 4 YEARS - 19 YEARS'],[nan]


In [78]:
def process_string_to_dict(string):
    string = string.split('; Count')[0]
    # Substituir os `;` por vírgulas e remover espaços extras
    formatted_string = re.sub(r";", ",", string)
    formatted_string = re.sub(r"=", ":", formatted_string)
    # Adicionar chaves externas para transformá-la em um dicionário
    formatted_string = "{" + formatted_string + "}"
    # Substituir chaves como "Code.or.Value" por "Code" e "Value.Description" por "Description"
    formatted_string = re.sub(r"Code\.or\.Value", '"Code"', formatted_string)
    formatted_string = re.sub(r"Value\.Description", '"Description"', formatted_string)

    return eval(formatted_string)

In [82]:
rows = []
for _, row in df_codebook_final.iterrows():
    # Ignorar linhas onde Tabela_Valores é NaN
    if pd.isna(row['Tabela_Valores']):
        rows.append({
                    "Column": row["Variable_Name"],
                    "Code": None,
                    "Label": None,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
        continue
    
    # Extrair os dados de Code e Value.Description
    valores = row['Tabela_Valores']
    
    print()
    print(valores)
    #print(valores.replace("\n", ""))
    valores = re.sub(r'\s+', ' ', valores)
    valores = re.sub(r':\s+\[', ': [', valores)
    valores = valores.replace('"', "'")
    
    print(valores)
    
    # Usando ast.literal_eval para transformar a string em dicionário
    parsed_values = process_string_to_dict(valores)
    codes = parsed_values["Code"]
    codes = [str(item).title() for item in codes]
    descriptions = parsed_values["Description"]
    descriptions = [str(item).title() for item in descriptions]
    # Criar uma nova linha para cada combinação de Code e Description
    print()
    print("********************************")
    print(codes)
    print(descriptions)
    print("********************************")
    print(row)
    
    for code, description in zip(codes, descriptions):
        rows.append({
            "Column": row["Variable_Name"],
            "Code": code,
            "Label": description,
            "English_Text": row["English_Text"],
            "Target": row["Target"],
            "English_Instructions": row["English_Instructions"]
        })


# Criar um novo DataFrame com os dados processados
df_expanded = pd.DataFrame(rows)
df_expanded['Class'] = None
df_expanded['Other For'] = None


Code.or.Value=["1", "2", "7", "9", "."]; Value.Description=["Yes", "No", "Refused", "Don't know", "Missing"]; Count=["2361", "7811", "0", "23", "5365"]; Cumulative=["2361", "10172", "10172", "10195", "15560"]; Skip.to.Item=["", "DBQ930", "DBQ930", "DBQ930", ""]
Code.or.Value=['1', '2', '7', '9', '.']; Value.Description=['Yes', 'No', 'Refused', 'Don't know', 'Missing']; Count=['2361', '7811', '0', '23', '5365']; Cumulative=['2361', '10172', '10172', '10195', '15560']; Skip.to.Item=['', 'DBQ930', 'DBQ930', 'DBQ930', '']


SyntaxError: unterminated string literal (detected at line 1) (<string>, line 1)

In [21]:
df_expanded

,Column,Code,Label,English_Text,Target,English_Instructions,Class,Other For
0,SEQN,None,None,['Respondent sequence number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
1,SDDSRVYR,1,Nhanes 1999-2000 Public Release,['Data Release Number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
2,SDDSRVYR,.,Missing,['Data Release Number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
3,RIDSTATR,1,Interviewed Only,['Interview and Examination Status of the Samp...,['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
4,RIDSTATR,2,Both Interviewed And Mec Examined,['Interview and Examination Status of the Samp...,['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
...,...,...,...,...,...,...,...,...
414,WTIREP50,.,Missing,['Interview Weight Jack Knife Replicate 50'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
415,WTIREP51,0 To 236648.97986,Range Of Values,['Interview Weight Jack Knife Replicate 51'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
416,WTIREP51,.,Missing,['Interview Weight Jack Knife Replicate 51'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
417,WTIREP52,0 To 236964.16066,Range Of Values,['Interview Weight Jack Knife Replicate 52'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None


In [22]:
df_expanded.drop_duplicates(inplace=True)

In [23]:
df_expanded

,Column,Code,Label,English_Text,Target,English_Instructions,Class,Other For
0,SEQN,None,None,['Respondent sequence number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
1,SDDSRVYR,1,Nhanes 1999-2000 Public Release,['Data Release Number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
2,SDDSRVYR,.,Missing,['Data Release Number.'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
3,RIDSTATR,1,Interviewed Only,['Interview and Examination Status of the Samp...,['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
4,RIDSTATR,2,Both Interviewed And Mec Examined,['Interview and Examination Status of the Samp...,['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
...,...,...,...,...,...,...,...,...
414,WTIREP50,.,Missing,['Interview Weight Jack Knife Replicate 50'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
415,WTIREP51,0 To 236648.97986,Range Of Values,['Interview Weight Jack Knife Replicate 51'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
416,WTIREP51,.,Missing,['Interview Weight Jack Knife Replicate 51'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None
417,WTIREP52,0 To 236964.16066,Range Of Values,['Interview Weight Jack Knife Replicate 52'],['Both males and females 0 YEARS - 150 YEARS'],[nan],None,None


In [18]:
df_expanded.to_csv('DEMO_codebook.csv', sep=',', encoding='utf-8')

In [23]:
dictionary_mapping

,Variable_Name,Attribute,attributeOf,Unit,Time,Entity,Role,Relation,inRelationTo,wasDerivedFrom,wasGeneratedBy
0,SEQN,None,None,None,None,None,None,None,None,None,None
1,SDDSRVYR,None,None,None,None,None,None,None,None,None,None
2,RIDSTATR,None,None,None,None,None,None,None,None,None,None
3,RIDEXMON,None,None,None,None,None,None,None,None,None,None
4,RIAGENDR,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
174,DMDMARTZ,None,None,None,None,None,None,None,None,None,None
175,DMDYRUSZ,None,None,None,None,None,None,None,None,None,None
176,WTINTPRP,None,None,None,None,None,None,None,None,None,None
177,WTMECPRP,None,None,None,None,None,None,None,None,None,None
